In [1]:
from pathlib import Path
import pandas as pd
from PIL import Image
from weavingtools.annotation_tools import *
from weavingtools.linkage_tools import *
from weavingtools.embedding_tools import *
import seaborn as sns
import numpy as np
sns.set_theme()

In [2]:
collection_df = pd.read_csv('data/heritage_weaver_data.csv')
collection_df  = collection_df[~collection_df['name'].isna()].reset_index(drop=True)
collection_df.shape


(28400, 12)

In [3]:
def extract_root_noun_and_adjectives(text: str) -> Tuple[str, List[str]]:
    """
    Extract the root noun and its dependent adjectives from a given text.

    Parameters:
    text (str): The input text.

    Returns:
    Tuple[str, List[str]]: A tuple containing the root noun and a list of its dependent adjectives.
    """
    doc = nlp(text)

    if len(doc) == 1:
        return doc[0].lemma_
    root_noun = ''
    quantifiers = []

    for token in doc:
        #print(token.text, token.dep_, token.pos_)
        if token.dep_ == 'ROOT' and token.pos_ in ['NOUN', 'PROPN']:
            root_noun = token.lemma_
            # for child in token.children:
            #     if (child.dep_ == 'amod' and child.pos_ == 'ADJ') or(child.dep_ == 'compound' and child.pos_ == 'NOUN'):
            #         quantifiers.append(child.lemma_)
            # break
        
    if root_noun == '':
        for token in doc:
            if token.pos_ in ['NOUN', 'PROPN']:
                root_noun = token.lemma_
                # for child in token.children:
                #     if (child.dep_ == 'amod' and child.pos_ == 'ADJ') or(child.dep_ == 'compound' and child.pos_ == 'NOUN'):
                #         quantifiers.append(child.lemma_)
                # break


    return  root_noun.lower()

# Example usage
text = "specimen"
output = extract_root_noun_and_adjectives(text)
output

''

In [4]:
import spacy
from tqdm import tqdm
tqdm.pandas()
nlp = spacy.load("en_core_web_lg")

def extract_head_nouns(text: str) -> List[str]:
    #text = text.replace('/ model', '').replace(', model', '').replace('- model', '')
    #doc = nlp(text)
    #head_nouns = [token.lemma_ for token in doc if token.dep_ == 'ROOT' and token.pos_ in ['NOUN','PROPN']]
    quantifiers, root_noun = extract_root_noun_and_adjectives(text)
    return quantifiers, root_noun

#collection_df[['quantifiers','noun_phrases']] = collection_df['name'].progress_apply(extract_head_nouns)
#collection_df['quantifiers'], collection_df['root_noun'] = list(zip(*collection_df['name'].progress_apply(extract_root_noun_and_adjectives)))
collection_df['root_noun'] = collection_df['name'].progress_apply(extract_root_noun_and_adjectives)

100%|██████████| 28400/28400 [01:05<00:00, 432.08it/s]


In [10]:
collection_df['root_noun'].value_counts().head(20)[:100].index

Index(['specimen', 'telephone', 'lamp', 'block', 'receiver', 'instrument',
       'tool', 'model', 'cable', 'meter', 'engine', 'sample', 'machine',
       'representation', 'photograph', 'component', 'telegraph', 'box',
       'thermometer', 'equipment'],
      dtype='object', name='root_noun')

In [5]:
with open('data/eval/eval_ids.txt','r') as f:
    eval_ids = f.read().splitlines()
print(len(eval_ids))

2172


In [6]:
eval_df = collection_df[collection_df['record_id'].isin(eval_ids)].reset_index(drop=True)
eval_df.shape

(1922, 13)

In [12]:
n_candidates = 10 # 5 or 10
target_col = 'root_noun' # 'root_noun' or 'name'

def add_names(row, df, column): # first_noun_phrase or name
    names = [row[column]]
    while len(names) < n_candidates:
        name = df[column].sample().values[0]
        if name not in names:
            names.append(name)

    return names


eval_df['candidate_labels'] = eval_df.apply(lambda x: add_names(x, collection_df, target_col), axis=1)


In [14]:
eval_df_subs = eval_df.sample(n=200, random_state=42).reset_index(drop=True)

In [15]:
from transformers import pipeline
from PIL import Image
from tqdm import tqdm
import requests
tqdm.pandas()

def get_predictions(row):
    image = Image.open(row['img_path'])
    predictions = detector(image, candidate_labels=row['candidate_labels'], max_length=512)
    return predictions

for ext, checkpoint in { 
                    'ft-best':'Kaspar/siglip-heritage-weaver-best',
                    'ft-last':'Kaspar/siglip-heritage-weaver-last',
                    'base':'google/siglip-base-patch16-224'
                    }.items():
    detector = pipeline(model=checkpoint, task="zero-shot-image-classification")
    eval_df_subs[f'predictions_{ext}'] = eval_df_subs.progress_apply(get_predictions, axis=1)


100%|██████████| 200/200 [01:09<00:00,  2.88it/s]


In [28]:
def get_label(predictions):
    label, max_score = '', 0
    for pred in predictions:
        if pred['score'] > max_score:
            label = pred['label']
            max_score = pred['score']
    return label

for ext in ['base','ft-last','ft-best']:
    eval_df_subs[f'label_{ext}'] = eval_df_subs[f'predictions_{ext}'].apply(lambda x: get_label(x))

In [32]:
from openai import OpenAI
import base64

client = OpenAI()

def classify_image(image_path: str, labels: list[str]) -> str:
    """
    Classify an image into one of the given labels using OpenAI Vision API.

    Args:
        image_path (str): Local path to the image.
        labels (list[str]): List of possible labels.

    Returns:
        str: Predicted label.
    """
    # Read image file and encode to base64
    with open(image_path, "rb") as image_file:
        base64_image = base64.b64encode(image_file.read()).decode("utf-8")

    # Prepare the system prompt and user prompt
    system_message = {
        "role": "system",
        "content": "You are a helpful assistant that classifies images into specific categories."
    }

    user_message = {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": f"Classify this image into one of these labels: {', '.join(labels)}. "
                        "Respond only with the label."
            },
            {
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/jpeg;base64,{base64_image}"
                }
            }
        ]
    }

    # Call the API
    response = client.chat.completions.create(
        model="gpt-4o",  # or "gpt-4o" or any vision-capable model
        messages=[system_message, user_message]
    )

    # Extract label from response
    label = response.choices[0].message.content.strip()
    return label



In [33]:
chatgpt_predictions = []
for i, row in tqdm(eval_df_subs.iterrows()):
    chatgpt_predictions.append(classify_image(row['img_path'], row['candidate_labels']))


200it [04:45,  1.43s/it]


In [37]:
eval_df_subs.columns

Index(['Unnamed: 0', 'record_id', 'name', 'description', 'category',
       'taxonomy', 'img_loc', 'img_name', 'img_path', 'downloaded', 'img_url',
       'collection', 'root_noun', 'candidate_labels', 'predictions_ft-best',
       'predictions_ft-last', 'predictions_base', 'predictions_chatgpt',
       'label_base', 'label_ft-last', 'label_ft-best', 'label_chatgpt',
       'prediction_base', 'prediction_ft-last', 'prediction_ft-best',
       'prediction_chatgpt', 'label_chatgpt_4o'],
      dtype='object')

In [42]:
eval_df_subs[f'label_chatgpt'].unique()

array(['specimen', 'cup', 'stitching', 'racket', 'directory', 'index',
       'bearing', 'slag', 'generator', 'transceiver', 'instrument',
       'sample', 'lamp', 'telephone', 'coal', 'meter', 'prop', 'oil',
       'bowl', 'tool', 'part', 'machine', 'ammeter', 'dish', 'mould',
       'cover', 'component', 'chip', 'model', 'container', 'protector',
       'bulb', 'ore', 'television', 'block', 'None of the labels apply.',
       'button', 'voltmeter', 'radio', 'probe', 'transmitter', 'ornament',
       'driver', 'tube', 'thermometer', 'magnet', 'seal', 'gauge',
       'dynamo', 'chisel', 'object', 'spanner', 'transistor', 'telegraph',
       'cork', 'strip', 'handlamp', 'badge', 'washstand', 'harness',
       'cassette', 'scoop', 'type', 'ratemeter', 'detector', 'copper',
       'photograph', 'nail', 'razor', 'head', 'box', 'diagram',
       'document', 'booklet', 'bobbin', 'dosemeter', 'tester', 'antenna',
       'tin', 'moulding', 'skewer', 'board', 'former', 'unit', 'cooker',
       

In [43]:
for ext in ['base','ft-last','ft-best', 'chatgpt','chatgpt_4o']:
    eval_df_subs[f'prediction_{ext}'] = eval_df_subs[f'label_{ext}'].str.lower() == eval_df_subs[target_col] #root_noun | name
    print(ext, ' ',round(eval_df_subs[f'prediction_{ext}'].mean(),4))

base   0.635
ft-last   0.805
ft-best   0.825
chatgpt   0.66
chatgpt_4o   0.76


In [ ]:
# 200/200 [01:09<00:00,  2.86it/s]
# 200/200 [01:07<00:00,  2.96it/s]
# 200/200 [01:09<00:00,  2.88it/s]
# 200it [04:28,  1.34s/it]
# 200it [04:45,  1.43s/it]

In [78]:
all_labels = set([l for row in eval_df_subs['candidate_labels'] for l in row])
lab2idx = {l:i for i,l in enumerate(sorted(all_labels))}
lab2idx['none of the labels apply'] = len(all_labels)


In [79]:
lab2idx

{'': 0,
 'Bitumen': 1,
 'Stereoscope': 2,
 'accelerator': 3,
 'accessory': 4,
 'accumulator': 5,
 'adaptor': 6,
 'aerial': 7,
 'aid': 8,
 'album': 9,
 'alloy': 10,
 'ammeter': 11,
 'amplifier': 12,
 'analyser': 13,
 'anemometer': 14,
 'antenna': 15,
 'apparatus': 16,
 'arc': 17,
 'artefact': 18,
 'ashtray': 19,
 'attachment': 20,
 'auger': 21,
 'awl': 22,
 'badge': 23,
 'bag': 24,
 'balance': 25,
 'bangle': 26,
 'basalt': 27,
 'basket': 28,
 'battery': 29,
 'beaker': 30,
 'bearing': 31,
 'bell': 32,
 'binder': 33,
 'bit': 34,
 'blade': 35,
 'block': 36,
 'blowpipe': 37,
 'board': 38,
 'bobbin': 39,
 'boiler': 40,
 'book': 41,
 'booklet': 42,
 'bottle': 43,
 'bow': 44,
 'bowl': 45,
 'box': 46,
 'bracket': 47,
 'bradal': 48,
 'breaker': 49,
 'brick': 50,
 'bronze': 51,
 'brooch': 52,
 'brush': 53,
 'bucket': 54,
 'bulb': 55,
 'burner': 56,
 'bust': 57,
 'button': 58,
 'cable': 59,
 'calculator': 60,
 'caliper': 61,
 'calorimeter': 62,
 'camera': 63,
 'candleholder': 64,
 'candlestick': 6

In [80]:
eval_df_subs['pred'].unique()

array([399, 105, 412, 328, 118, 194, 31, 389, 163, 443, 198, 358, 213,
       426, 85, 244, 321, 278, 45, 439, 290, 229, 11, 119, 261, 103, 90,
       75, 256, 96, 323, 55, 279, 427, 36, 'none of the labels apply', 58,
       469, 329, 319, 446, 280, 127, 453, 431, 232, 365, 162, 130, 76,
       276, 398, 445, 425, 100, 414, 178, 23, 473, 181, 68, 363, 459, 332,
       111, 99, 299, 267, 333, 182, 46, 113, 121, 42, 39, 122, 430, 15,
       437, 262, 387, 38, 151, 461, 98, 136, 335, 310, 247, 465, 147],
      dtype=object)

In [81]:
from sklearn.metrics import classification_report, accuracy_score, precision_score, f1_score
from collections import defaultdict
results = defaultdict(dict)
for ext in ['base','ft-last','ft-best', 'chatgpt','chatgpt_4o']:
    #eval_df[f'prediction_{ext}'] = eval_df[f'label_{ext}'] == eval_df['root_noun'] #root_noun | name
    eval_df_subs['pred'] = eval_df_subs[f'label_{ext}'].str.lower().str.rstrip('.').replace(lab2idx)
    eval_df_subs['true'] = eval_df_subs[target_col].replace(lab2idx)
    results[ext]['accuracy'] = accuracy_score(eval_df_subs['pred'] , eval_df_subs['true'])
    results[ext]['precision'] = precision_score(eval_df_subs['pred'] , eval_df_subs['true'], average='weighted')
    results[ext]['f1'] = f1_score(eval_df_subs['pred'] , eval_df_subs['true'], average='weighted')
    

/var/folders/d2/ydv0grbd38985h6_95t0vdjw0000gp/T/ipykernel_49370/4191935842.py:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  eval_df_subs['pred'] = eval_df_subs[f'label_{ext}'].str.lower().str.rstrip('.').replace(lab2idx)
/var/folders/d2/ydv0grbd38985h6_95t0vdjw0000gp/T/ipykernel_49370/4191935842.py:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  eval_df_subs['true'] = eval_df_subs[target_col].replace(lab2idx)
/Users/kasparbeelen/anaconda3/envs/heritageweaver/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1509:

In [ ]:
res_df = pd.DataFrame(results)
res_df = res_df.round(3).T
res_df['target_col'] = target_col
res_df['n_candidates'] = n_candidates
print(res_df[['target_col' , 'n_candidates' ,'accuracy', 'precision','f1']].to_latex())

\begin{tabular}{llrrrr}
\toprule
 & target_col & n_candidates & accuracy & precision & f1 \\
\midrule
base & root_noun & 10 & 0.635000 & 0.707000 & 0.656000 \\
ft-last & root_noun & 10 & 0.805000 & 0.844000 & 0.811000 \\
ft-best & root_noun & 10 & 0.825000 & 0.836000 & 0.825000 \\
chatgpt & root_noun & 10 & 0.660000 & 0.729000 & 0.672000 \\
chatgpt_4o & root_noun & 10 & 0.770000 & 0.801000 & 0.771000 \\
\bottomrule
\end{tabular}

